# Significance Testing

Stage 4 artifact reader. The reusable implementations live in `src/significance.py`; saved tables are generated by `scripts/run_significance.py`.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))

import config
from scripts.run_significance import generate_significance_artifacts

results_dir = Path(config.DATA_RESULTS_DIR)
features_dir = Path(config.DATA_FEATURES_DIR)

## Refresh Artifacts

In [2]:
paths = generate_significance_artifacts(
    results_dir=results_dir,
    features_dir=features_dir,
    block_size=8,
    n_bootstrap=5000,
    seed=42,
)
pd.DataFrame({'artifact': paths.keys(), 'path': [str(p) for p in paths.values()]})

,artifact,path
0,weekly_errors,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
1,dm_results,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
2,macro_dm_results,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
3,bootstrap_results,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
4,macro_bootstrap_results,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
5,summary,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...
6,macro_summary,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...


## Weekly Model Errors

In [3]:
weekly_errors = pd.read_parquet(results_dir / 'weekly_model_errors.parquet')
weekly_error_summary = (
    weekly_errors.groupby('model')
    .agg(mean_weekly_mse=('weekly_mse', 'mean'), n_weeks=('week', 'nunique'), min_stocks=('n_stocks', 'min'))
    .sort_values('mean_weekly_mse')
)
display(weekly_error_summary)
display(weekly_errors.head(12))

,mean_weekly_mse,n_weeks,min_stocks
model,,,
GNN-Correlation + Macro 21d,0.029575,103,465
GNN-Correlation + Macro Tuned,0.030889,103,465
GNN-Correlation + Macro 126d,0.031029,103,465
GNN-Granger-Volatility + Macro,0.031375,103,465
GNN-Granger + Macro,0.031439,103,465
GNN-Sector + Macro,0.031508,103,465
GNN-Ensemble + Macro,0.031598,103,465
GNN-Ensemble,0.032012,103,465
GNN-Correlation + Macro 63d,0.032187,103,465


,week,model,experiment_id,model_family,graph_type,loss_type,feature_version,graph_version,prediction_path,weekly_mse,n_stocks
0,2024-01-01,GNN-Correlation,baseline_gnn_correlation,GNN,correlation,mse,stock_features_v1,correlation_threshold_0.3_lookback_252,data/results/test_preds_gnn_corr.parquet,0.017233,465
1,2024-01-01,GNN-Correlation + Macro,macro_gnn_correlation,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_252,data/results/test_preds_gnn_corr_macro.parquet,0.015244,465
2,2024-01-01,GNN-Correlation + Macro 126d,window_gnn_correlation_macro_126,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_126,data/results/test_preds_gnn_corr_macro_w126.pa...,0.015754,465
3,2024-01-01,GNN-Correlation + Macro 21d,window_gnn_correlation_macro_21,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_21,data/results/test_preds_gnn_corr_macro_w021.pa...,0.013761,465
4,2024-01-01,GNN-Correlation + Macro 63d,window_gnn_correlation_macro_63,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_63,data/results/test_preds_gnn_corr_macro_w063.pa...,0.014210,465
5,2024-01-01,GNN-Correlation + Macro Tuned,macro_gnn_correlation_hparam,GNN,correlation,mse,stock_features_plus_regime_v1,correlation_threshold_0.3_lookback_252,data/results/test_preds_gnn_corr_macro_hparam....,0.014660,465
6,2024-01-01,GNN-Ensemble,baseline_gnn_ensemble,GNN ensemble,correlation+sector+granger,mse,stock_features_v1,correlation+sector+granger_v1,data/results/test_preds_gnn_ensemble.parquet,0.018567,465
7,2024-01-01,GNN-Ensemble + Macro,macro_gnn_ensemble,GNN ensemble,ensemble,mse,stock_features_plus_regime_v1,correlation+sector+granger_v1,data/results/test_preds_gnn_ensemble_macro.par...,0.015101,465
8,2024-01-01,GNN-Granger,baseline_gnn_granger,GNN,granger,mse,stock_features_v1,granger_lag_5_bonferroni,data/results/test_preds_gnn_granger.parquet,0.020228,465
9,2024-01-01,GNN-Granger + Macro,macro_gnn_granger,GNN,granger,mse,stock_features_plus_regime_v1,granger_lag_5_bonferroni,data/results/test_preds_gnn_granger_macro.parquet,0.015074,465


## Diebold-Mariano Tests

In [4]:
dm_results = pd.read_csv(results_dir / 'dm_test_results.csv')
dm_view = dm_results.sort_values(['baseline', 'p_value_bh', 'p_value'])
display(dm_view)
display(dm_view[dm_view['rejected_bh']])

,model,baseline,dm_stat,p_value,n_weeks,mean_loss_diff,p_value_bh,rejected_bh
0,GNN-Correlation + Macro Tuned,HAR per-stock,3.220623,0.000858,103,0.001969,0.015449,True
1,GNN-Correlation + Macro 21d,HAR per-stock,2.178505,0.015836,103,0.003283,0.130042,False
2,GNN-Granger + Macro,HAR per-stock,1.938683,0.027652,103,0.001419,0.130042,False
3,GNN-Correlation + Macro 126d,HAR per-stock,1.918867,0.028898,103,0.001829,0.130042,False
4,GNN-Ensemble + Macro,HAR per-stock,1.748051,0.041732,103,0.001260,0.130496,False
5,GNN-Granger-Volatility + Macro,HAR per-stock,1.728096,0.043499,103,0.001483,0.130496,False
6,GNN-Sector + Macro,HAR per-stock,1.344909,0.090819,103,0.001350,0.233534,False
7,GNN-Ensemble,HAR per-stock,0.716666,0.237609,103,0.000846,0.503786,False
8,GNN-Correlation,HAR per-stock,0.646150,0.259817,103,0.000667,0.503786,False
9,GNN-Correlation + Macro 63d,HAR per-stock,0.585115,0.279881,103,0.000671,0.503786,False


,model,baseline,dm_stat,p_value,n_weeks,mean_loss_diff,p_value_bh,rejected_bh
0,GNN-Correlation + Macro Tuned,HAR per-stock,3.220623,0.000858,103,0.001969,0.015449,True
18,GNN-Correlation + Macro Tuned,HAR pooled,3.594487,0.000251,103,0.002216,0.004523,True


## Bootstrap Sharpe Confidence Intervals

In [5]:
bootstrap_ci = pd.read_csv(results_dir / 'bootstrap_sharpe_ci.csv')
display(
    bootstrap_ci[bootstrap_ci['comparison'] == 'sharpe']
    .sort_values(['strategy', 'point_estimate'], ascending=[True, False])
)
display(
    bootstrap_ci[bootstrap_ci['comparison'] == 'sharpe_diff']
    .sort_values(['strategy', 'ci_lower'], ascending=[True, False])
)

,strategy,return_path,model,comparison,benchmark,point_estimate,ci_lower,ci_upper,n_weeks,block_size,n_bootstrap
0,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,Equal-weight,sharpe,NaN,0.878126,-0.201938,2.166350,103,8,5000
12,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Sector + Macro,sharpe,NaN,0.855936,-0.216442,2.125156,103,8,5000
8,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Ensemble + Macro,sharpe,NaN,0.849768,-0.221979,2.124832,103,8,5000
2,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation + Macro,sharpe,NaN,0.843833,-0.221948,2.112744,103,8,5000
10,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Granger + Macro,sharpe,NaN,0.838265,-0.236175,2.128808,103,8,5000
...,...,...,...,...,...,...,...,...,...,...,...
80,volatility_targeted,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,LSTM + Macro,sharpe,NaN,0.687001,-0.386660,1.959097,103,8,5000
65,volatility_targeted,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation,sharpe,NaN,0.649099,-0.400606,1.871651,103,8,5000
69,volatility_targeted,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation + Macro 63d,sharpe,NaN,0.557123,-0.490578,1.757772,103,8,5000
70,volatility_targeted,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation + Macro Tuned,sharpe,NaN,0.539924,-0.504194,1.768028,103,8,5000


,strategy,return_path,model,comparison,benchmark,point_estimate,ci_lower,ci_upper,n_weeks,block_size,n_bootstrap
26,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Granger + Macro,sharpe_diff,Equal-weight,-0.039862,-0.123562,0.054643,103,8,5000
24,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Ensemble + Macro,sharpe_diff,Equal-weight,-0.028359,-0.128537,0.076046,103,8,5000
18,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation + Macro,sharpe_diff,Equal-weight,-0.034294,-0.137047,0.069839,103,8,5000
28,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Sector + Macro,sharpe_diff,Equal-weight,-0.022190,-0.139047,0.096028,103,8,5000
25,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Granger,sharpe_diff,Equal-weight,-0.066998,-0.189827,0.055078,103,8,5000
32,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,LSTM + Macro,sharpe_diff,Equal-weight,-0.071252,-0.189996,0.058293,103,8,5000
20,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Correlation + Macro 21d,sharpe_diff,Equal-weight,-0.056922,-0.192336,0.079418,103,8,5000
23,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,GNN-Ensemble,sharpe_diff,Equal-weight,-0.071736,-0.206193,0.064980,103,8,5000
31,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,LSTM,sharpe_diff,Equal-weight,-0.072406,-0.206696,0.073783,103,8,5000
30,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics...,HAR pooled,sharpe_diff,Equal-weight,-0.080338,-0.222660,0.063997,103,8,5000


## Significance Summary

In [6]:
summary = pd.read_csv(results_dir / 'significance_summary.csv')
display(summary)

,section,metric,value,details
0,dm_tests,fdr_significant_model_vs_baseline,2.000000,54 DM comparisons at FDR 0.05
1,dm_tests,min_bh_adjusted_p,0.004523,One-sided lower-loss alternative
2,bootstrap,positive_sharpe_diff_ci,0.000000,16 Sharpe-difference intervals vs available be...


# STEP7_MACRO_EVAL_START
## Step 7 - Macro Feature Significance Tests

The refresh step above writes both all-model significance artifacts and matched macro-vs-baseline significance artifacts.

In [7]:
macro_dm = pd.read_csv(results_dir / 'macro_dm_test_results.csv')
macro_bootstrap = pd.read_csv(results_dir / 'macro_bootstrap_sharpe_ci.csv')
macro_summary = pd.read_csv(results_dir / 'macro_significance_summary.csv')

print('Matched macro-vs-baseline DM tests')
display(macro_dm.style.format(precision=6))

print('Matched macro-vs-baseline Sharpe difference bootstrap intervals')
display(macro_bootstrap.style.format(precision=6))

print('Macro significance summary')
display(macro_summary)

Matched macro-vs-baseline DM tests


,model,baseline,dm_stat,p_value,n_weeks,mean_loss_diff,p_value_bh,rejected_bh
0,GNN-Granger + Macro,GNN-Granger,1.854058,0.033310,103,0.002263,0.299791,False
1,GNN-Correlation + Macro Tuned,GNN-Correlation,1.506225,0.067550,103,0.001302,0.299866,False
2,GNN-Correlation + Macro 21d,GNN-Correlation,1.290164,0.099955,103,0.002615,0.299866,False
3,GNN-Sector + Macro,GNN-Sector,1.112589,0.134250,103,0.002123,0.302063,False
4,GNN-Correlation + Macro 126d,GNN-Correlation,0.940804,0.174514,103,0.001162,0.314125,False
5,GNN-Ensemble + Macro,GNN-Ensemble,0.364250,0.358213,103,0.000414,0.537319,False
6,GNN-Correlation + Macro 63d,GNN-Correlation,0.008665,0.496552,103,0.000003,0.638423,False
7,LSTM + Macro,LSTM,-0.596690,0.723982,103,-0.002421,0.814479,False
8,GNN-Correlation + Macro,GNN-Correlation,-2.980684,0.998202,103,-0.001192,0.998202,False


Matched macro-vs-baseline Sharpe difference bootstrap intervals


,strategy,return_path,model,baseline,comparison,point_estimate,ci_lower,ci_upper,n_weeks,block_size,n_bootstrap
0,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Correlation + Macro,GNN-Correlation,macro_sharpe_diff,0.065489,0.018878,0.117344,103,8,5000
1,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Correlation + Macro 126d,GNN-Correlation,macro_sharpe_diff,0.031548,-0.022509,0.084012,103,8,5000
2,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Correlation + Macro 21d,GNN-Correlation,macro_sharpe_diff,0.042860,0.006306,0.079886,103,8,5000
3,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Correlation + Macro 63d,GNN-Correlation,macro_sharpe_diff,0.016774,-0.051403,0.071234,103,8,5000
4,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Correlation + Macro Tuned,GNN-Correlation,macro_sharpe_diff,0.047159,-0.021658,0.109931,103,8,5000
5,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Ensemble + Macro,GNN-Ensemble,macro_sharpe_diff,0.043377,0.004315,0.085674,103,8,5000
6,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Granger + Macro,GNN-Granger,macro_sharpe_diff,0.027137,-0.009915,0.075709,103,8,5000
7,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,GNN-Sector + Macro,GNN-Sector,macro_sharpe_diff,0.057775,-0.015548,0.148495,103,8,5000
8,long_only_inverse_vol,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_returns.parquet,LSTM + Macro,LSTM,macro_sharpe_diff,0.001153,-0.027297,0.027133,103,8,5000
9,long_short,C:\Users\Rylan Wade\Desktop\FinancialAnalytics\Final_Project\data\results\portfolio_ls_returns.parquet,GNN-Correlation + Macro,GNN-Correlation,macro_sharpe_diff,0.657367,0.366168,1.050734,103,8,5000


Macro significance summary


,section,metric,value,details
0,macro_dm_tests,matched_pairs_fdr_significant,0.000000,9 matched macro-vs-baseline DM comparisons
1,macro_dm_tests,min_bh_adjusted_p,0.299791,One-sided lower-loss alternative for macro mod...
2,macro_bootstrap,positive_sharpe_diff_ci,10.000000,36 matched Sharpe-difference intervals


# STEP7_MACRO_EVAL_END